In [7]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    f.write('{"username":"schwarzam","key":""}')

os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download anokas/kuzushiji
!unzip -o kuzushiji.zip "kmnist*.npz"

Dataset URL: https://www.kaggle.com/datasets/anokas/kuzushiji
License(s): CC-BY-SA-4.0
100% 571M/571M [00:03<00:00, 152MB/s]

Archive:  kuzushiji.zip
  inflating: kmnist-test-imgs.npz    
  inflating: kmnist-test-labels.npz  
  inflating: kmnist-train-imgs.npz   
  inflating: kmnist-train-labels.npz  


In [8]:
# kuzushiji_classificacao.py
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ============================================================
# 1) Carregamento do dataset
# ============================================================
AX = np.load("kmnist-train-imgs.npz")["arr_0"]
AY = np.load("kmnist-train-labels.npz")["arr_0"]
QX = np.load("kmnist-test-imgs.npz")["arr_0"]
QY = np.load("kmnist-test-labels.npz")["arr_0"]

print("Kuzushiji-MNIST carregado com sucesso!")
print("Shape de AX:", AX.shape)
print("Shape de AY:", AY.shape)
print("Shape de QX:", QX.shape)
print("Shape de QY:", QY.shape)

Kuzushiji-MNIST carregado com sucesso!
Shape de AX: (60000, 28, 28)
Shape de AY: (60000,)
Shape de QX: (10000, 28, 28)
Shape de QY: (10000,)


In [9]:

# ============================================================
# 2) Pré-processamento
# ============================================================
# normalização para [0,1]
AX_densa = AX.astype("float32") / 255.0
QX_densa = QX.astype("float32") / 255.0

# para rede densa: achata 28x28 -> 784
AX_densa = AX_densa.reshape(-1, 28 * 28)
QX_densa = QX_densa.reshape(-1, 28 * 28)

# para rede convolucional: mantém 28x28 e adiciona canal
AX_cnn = (AX.astype("float32") / 255.0)[..., np.newaxis]
QX_cnn = (QX.astype("float32") / 255.0)[..., np.newaxis]

print("Shape AX_densa:", AX_densa.shape)
print("Shape AX_cnn:", AX_cnn.shape)

Shape AX_densa: (60000, 784)
Shape AX_cnn: (60000, 28, 28, 1)


In [10]:
# ============================================================
# 3) Função de perda e métrica
# ============================================================
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)


In [11]:

# ============================================================
# 4) Modelo denso
# ============================================================
def build_dense_model():
    model = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(512, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(128, activation="relu"),
        # sem softmax
        layers.Dense(10, activation="linear")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss=loss_fn,
        metrics=["accuracy"]
    )
    return model

In [12]:
# ============================================================
# 5) Modelo convolucional
# ============================================================
def build_cnn_model():
    model = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),

        layers.Conv2D(32, kernel_size=3, activation="relu", padding="same"),
        layers.Conv2D(32, kernel_size=3, activation="relu", padding="same"),
        layers.MaxPooling2D(pool_size=2),
        layers.Dropout(0.25),

        layers.Conv2D(64, kernel_size=3, activation="relu", padding="same"),
        layers.Conv2D(64, kernel_size=3, activation="relu", padding="same"),
        layers.MaxPooling2D(pool_size=2),
        layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),

        # sem softmax
        layers.Dense(10, activation="linear")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss=loss_fn,
        metrics=["accuracy"]
    )
    return model

In [13]:
# ============================================================
# 6) Treinamento e avaliação - rede densa
# ============================================================
dense_model = build_dense_model()
dense_model.summary()

t0 = time.perf_counter()
hist_dense = dense_model.fit(
    AX_densa, AY,
    validation_split=0.1,
    epochs=30,
    batch_size=100,
    verbose=2
)
t1 = time.perf_counter()

dense_score = dense_model.evaluate(QX_densa, QY, verbose=0)
dense_loss = dense_score[0]
dense_acc = dense_score[1]
dense_err = 100.0 * (1.0 - dense_acc)

print("\n=== RESULTADO REDE DENSA ===")
print(f"Test loss: {dense_loss:.4f}")
print(f"Test accuracy: {100*dense_acc:.2f}%")
print(f"Test error: {dense_err:.2f}%")
print(f"Tempo de treino: {t1 - t0:.2f} s")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │       401,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 567,434 (2.16 MB)

 Trainable params: 567,434 (2.16 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
540/540 - 10s - 18ms/step - accuracy: 0.8649 - loss: 0.4345 - val_accuracy: 0.9282 - val_loss: 0.2310
Epoch 2/30
540/540 - 8s - 15ms/step - accuracy: 0.9416 - loss: 0.1944 - val_accuracy: 0.9545 - val_loss: 0.1500
Epoch 3/30
540/540 - 11s - 20ms/step - accuracy: 0.9583 - loss: 0.1349 - val_accuracy: 0.9545 - val_loss: 0.1456
Epoch 4/30
540/540 - 7s - 14ms/step - accuracy: 0.9659 - loss: 0.1094 - val_accuracy: 0.9605 - val_loss: 0.1361
Epoch 5/30
540/540 - 8s - 15ms/step - accuracy: 0.9725 - loss: 0.0875 - val_accuracy: 0.9592 - val_loss: 0.1383
Epoch 6/30
540/540 - 8s - 16ms/step - accuracy: 0.9760 - loss: 0.0744 - val_accuracy: 0.9597 - val_loss: 0.1466
Epoch 7/30
540/540 - 7s - 13ms/step - accuracy: 0.9791 - loss: 0.0647 - val_accuracy: 0.9645 - val_loss: 0.1291
Epoch 8/30
540/540 - 11s - 20ms/step - accuracy: 0.9828 - loss: 0.0546 - val_accuracy: 0.9643 - val_loss: 0.1379
Epoch 9/30
540/540 - 8s - 15ms/step - accuracy: 0.9829 - loss: 0.0518 - val_accuracy: 0.9600 - val_lo

In [15]:
# ============================================================
# 7) Treinamento e avaliação - rede convolucional
# ============================================================
cnn_model = build_cnn_model()
cnn_model.summary()

t2 = time.perf_counter()
hist_cnn = cnn_model.fit(
    AX_cnn, AY,
    validation_split=0.1,
    epochs=6,
    batch_size=100,
    verbose=2
)
t3 = time.perf_counter()

cnn_score = cnn_model.evaluate(QX_cnn, QY, verbose=0)
cnn_loss = cnn_score[0]
cnn_acc = cnn_score[1]
cnn_err = 100.0 * (1.0 - cnn_acc)

print("\n=== RESULTADO REDE CONVOLUCIONAL ===")
print(f"Test loss: {cnn_loss:.4f}")
print(f"Test accuracy: {100*cnn_acc:.2f}%")
print(f"Test error: {cnn_err:.2f}%")
print(f"Tempo de treino: {t3 - t2:.2f} s")

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 28, 28, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 14, 14, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       803,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 870,634 (3.32 MB)

 Trainable params: 870,634 (3.32 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/6
540/540 - 226s - 419ms/step - accuracy: 0.8841 - loss: 0.3696 - val_accuracy: 0.9713 - val_loss: 0.0962
Epoch 2/6
540/540 - 216s - 400ms/step - accuracy: 0.9610 - loss: 0.1288 - val_accuracy: 0.9822 - val_loss: 0.0590
Epoch 3/6
540/540 - 268s - 497ms/step - accuracy: 0.9735 - loss: 0.0875 - val_accuracy: 0.9833 - val_loss: 0.0542
Epoch 4/6
540/540 - 213s - 395ms/step - accuracy: 0.9783 - loss: 0.0687 - val_accuracy: 0.9862 - val_loss: 0.0453
Epoch 5/6
540/540 - 218s - 404ms/step - accuracy: 0.9827 - loss: 0.0580 - val_accuracy: 0.9870 - val_loss: 0.0441
Epoch 6/6
540/540 - 217s - 403ms/step - accuracy: 0.9840 - loss: 0.0496 - val_accuracy: 0.9878 - val_loss: 0.0404

=== RESULTADO REDE CONVOLUCIONAL ===
Test loss: 0.1459
Test accuracy: 96.39%
Test error: 3.61%
Tempo de treino: 1359.76 s


In [16]:
# ============================================================
# 8) Comparação final
# ============================================================
print("\n=== COMPARAÇÃO FINAL ===")
print(f"Erro da rede densa:         {dense_err:.2f}%")
print(f"Erro da rede convolucional: {cnn_err:.2f}%")

if dense_err < cnn_err:
    print("A rede densa obteve a menor taxa de erro.")
elif cnn_err < dense_err:
    print("A rede convolucional obteve a menor taxa de erro.")
else:
    print("As duas redes obtiveram a mesma taxa de erro.")


=== COMPARAÇÃO FINAL ===
Erro da rede densa:         8.22%
Erro da rede convolucional: 3.61%
A rede convolucional obteve a menor taxa de erro.
